# Day 12–15 · Docker + Alembic + API Verification

**This notebook does NOT run Python code** (except curl-style HTTP calls).  
It walks you through every terminal command you need to:

1. Train and save the ML model
2. Start Docker containers
3. Run Alembic migrations (create DB tables)
4. Verify every API endpoint works
5. Run the test suite

**Run these commands in your terminal**, not inside Jupyter.

---
**Working directory for all commands:** `backend/`

## Step 1 · Train the ML model (run ONCE before Docker)

The model needs to be trained before the FastAPI server starts,  
because `main.py` tries to load it from `app/ml_models/*.pkl`.  

**Option A: Run the training notebook**  
Open `notebooks/04_model_training.ipynb` and run all cells.

**Option B: Run from terminal**
```bash
cd backend
pip install -r requirements.txt     # if not already done
python -m app.services.ml_model
```

**Expected output:**
```
Training food suitability ML model
  Food DB loaded: 590 items
  Training data shape: (5310, 14)
  Label distribution:
    safe   : 4890 (92.1%)
    caution:  333 ( 6.3%)
    avoid  :   87 ( 1.6%)
  5-Fold CV Accuracy: 0.98x ± 0.00x
  Saved: app/ml_models/food_suitability_model.pkl
```

After this, `app/ml_models/` should contain 3 files:
```
food_suitability_model.pkl
label_encoder.pkl
feature_names.json
```

## Step 2 · Copy CSV data files

All 4 dataset files must be in `backend/app/data/` before building Docker:

```bash
# From project root:
cp food_db_final_.csv                      backend/app/data/
cp Personalized_Diet_Recommendations.csv   backend/app/data/
cp detailed_meals_macros_CLEANED.csv       backend/app/data/
cp Food_and_Nutrition__.csv                backend/app/data/
```

Verify:
```bash
ls backend/app/data/
# Should show all 4 CSV files + the generated merged_training_data.csv
```

## Step 3 · Start Docker containers

```bash
cd backend
docker compose up --build
```

**What you should see:**
```
food_app_db   | database system is ready to accept connections
food_app_api  | ✓ Database tables ready
food_app_api  | ✓ Food database loaded: 590 items from food_db_final_.csv
food_app_api  | ✓ ML model loaded (13 features)
food_app_api  | ✓ Server ready. Visit http://localhost:8000/docs
food_app_api  | INFO: Uvicorn running on http://0.0.0.0:8000
```

**If FastAPI starts before postgres is ready:**  
The `depends_on: condition: service_healthy` in docker-compose.yml prevents this.  
If it still happens, just wait — Uvicorn will retry the DB connection.

**Run in background (detached mode):**
```bash
docker compose up --build -d
docker compose logs -f fastapi    # follow only the API logs
```

## Step 4 · Run Alembic migrations (first time only)

The `main.py` lifespan already calls `Base.metadata.create_all()` which creates  
tables on first run. Alembic is for FUTURE schema changes.  

Run this now to set Alembic up properly:

```bash
# Open shell inside running fastapi container
docker compose exec fastapi bash

# Inside the container — check current migration state
alembic current

# Stamp the DB as being at the initial migration
# (tables were created by create_all, so we just mark them as migrated)
alembic stamp head

# Verify
alembic history
# Should show:  a1b2c3d4e5f6 -> (head) initial tables

exit
```

**Adding a new column later (example workflow):**
```bash
# 1. Edit app/models.py — add your new column
# 2. Generate migration
docker compose exec fastapi alembic revision --autogenerate -m "add bmi to user_profiles"
# 3. Apply it
docker compose exec fastapi alembic upgrade head
# 4. Restart FastAPI to pick up model changes
docker compose restart fastapi
```

## Step 5 · Test all API endpoints

### Option A: Swagger UI (easiest)
Open http://localhost:8000/docs in your browser.  
Every endpoint is listed. Click "Try it out" → fill in values → Execute.

### Option B: curl commands

Run these in a terminal (all containers must be running):

In [ ]:
import httpx, asyncio, json

BASE = 'http://localhost:8000/api'

# ── Helper to pretty-print responses ────────────────────────────────────────
def show(label, resp):
    status_icon = '✓' if resp.status_code < 300 else '✗'
    print(f'\n{status_icon} {label} [{resp.status_code}]')
    try:
        data = resp.json()
        # Truncate long fields for readability
        if isinstance(data, dict):
            for k in ['nutrients', 'ingredient_matches']:
                if k in data and data[k]:
                    data[k] = f'<{len(str(data[k]))} chars — truncated>'
        print(json.dumps(data, indent=2)[:600])
    except Exception:
        print(resp.text[:300])

In [ ]:
# ── 1. Health check ──────────────────────────────────────────────────────────
r = httpx.get('http://localhost:8000/health')
show('GET /health', r)

In [ ]:
# ── 2. Create user profile ────────────────────────────────────────────────────
r = httpx.post(f'{BASE}/profile', json={
    'name':       'Ahmed Rahman',
    'age':        45,
    'gender':     'Male',
    'height_cm':  175,
    'weight_kg':  82,
    'diseases':   ['Hypertension', 'Diabetes'],
    'allergies':  ['Nut Allergy'],
    'activity_level': 'Moderately Active',
})
show('POST /api/profile', r)
USER_ID = r.json().get('id') if r.status_code == 201 else None
print(f'\n  → user_id = {USER_ID}  (save this for the next cells)')

In [ ]:
# Set your user_id here if you already created a profile
# USER_ID = 1

# ── 3. Get profile ────────────────────────────────────────────────────────────
r = httpx.get(f'{BASE}/profile/{USER_ID}')
show(f'GET /api/profile/{USER_ID}', r)

In [ ]:
# ── 4. Manual food check — safe ────────────────────────────────────────────────
# Scrambled eggs: low sodium, high protein → should be safe/caution for Hypertension
r = httpx.post(f'{BASE}/check-food', json={
    'user_id':   USER_ID,
    'food_name': 'Scrambled Eggs',
})
show('POST /api/check-food (Scrambled Eggs)', r)

In [ ]:
# ── 5. Manual food check — typo ───────────────────────────────────────────────
# Fuzzy matching should still find 'Grilled Chicken Salad'
r = httpx.post(f'{BASE}/check-food', json={
    'user_id':   USER_ID,
    'food_name': 'griled chiken salad',
})
show('POST /api/check-food (typo test)', r)
data = r.json()
if data.get('food_info'):
    print(f'  → Matched to: {data["food_info"]["food_item"]} (score={data["food_info"]["match_score"]})')

In [ ]:
# ── 6. OCR endpoint — allergy trigger ─────────────────────────────────────────
# This text contains 'almonds' → triggers Nut Allergy → hard block → avoid
r = httpx.post(f'{BASE}/analyze-ocr', json={
    'user_id':  USER_ID,
    'ocr_text': 'Ingredients: whole wheat flour (45%), sugar, palm oil, '
                'almonds (5%), skimmed milk powder, salt, E330.',
})
show('POST /api/analyze-ocr (nut allergy trigger)', r)
data = r.json()
print(f'  → Verdict: {data.get("verdict")} | Score: {data.get("score")}')
for w in data.get('warnings', []):
    print(f'  ⚠ {w}')

In [ ]:
# ── 7. OCR endpoint — no allergy ──────────────────────────────────────────────
r = httpx.post(f'{BASE}/analyze-ocr', json={
    'user_id':  USER_ID,
    'ocr_text': 'Ingredients: oats (60%), brown rice, corn flour, '
                'dried mango, salt, vitamin B12.',
})
show('POST /api/analyze-ocr (no allergy)', r)

In [ ]:
# ── 8. Image prediction endpoint ──────────────────────────────────────────────
# Simulates teammate's model predicting 'Banana' with 88% confidence
r = httpx.post(f'{BASE}/analyze-image', json={
    'user_id':    USER_ID,
    'food_label': 'Banana',
    'confidence': 0.88,
})
show('POST /api/analyze-image (Banana)', r)

In [ ]:
# ── 9. Image prediction — low confidence (should return 400) ─────────────────
r = httpx.post(f'{BASE}/analyze-image', json={
    'user_id':    USER_ID,
    'food_label': 'Banana',
    'confidence': 0.30,   # below 0.5 threshold
})
show('POST /api/analyze-image (low confidence)', r)
print(f'  → Expected 400, got {r.status_code}')

In [ ]:
# ── 10. History ────────────────────────────────────────────────────────────────
r = httpx.get(f'{BASE}/history/{USER_ID}')
show(f'GET /api/history/{USER_ID}', r)
history = r.json()
if history:
    print(f'  → {len(history)} checks recorded')
    for item in history[:3]:
        print(f'    [{item["verdict"]:7s}] {item["food_found"] or item["query"][:40]:40s} ({item["input_mode"]})')

In [ ]:
# ── 11. History detail ────────────────────────────────────────────────────────
if history:
    check_id = history[0]['id']
    r = httpx.get(f'{BASE}/history/{USER_ID}/{check_id}')
    show(f'GET /api/history/{USER_ID}/{check_id}', r)

In [ ]:
# ── 12. Delete history item ───────────────────────────────────────────────────
if history:
    check_id = history[-1]['id']
    r = httpx.delete(f'{BASE}/history/{check_id}')
    print(f'DELETE /api/history/{check_id} → {r.status_code}  (expected: 204)')

## Step 6 · Run the test suite

```bash
# Inside the fastapi container:
docker compose exec fastapi pytest app/tests/test_api.py -v

# Or locally (needs aiosqlite for SQLite test DB):
pip install aiosqlite pytest-asyncio
cd backend
pytest app/tests/test_api.py -v
```

**Expected output:**
```
PASSED app/tests/test_api.py::test_health
PASSED app/tests/test_api.py::test_create_profile
PASSED app/tests/test_api.py::test_get_profile
PASSED app/tests/test_api.py::test_get_profile_not_found
PASSED app/tests/test_api.py::test_update_profile
PASSED app/tests/test_api.py::test_invalid_disease_rejected
PASSED app/tests/test_api.py::test_check_food_safe
PASSED app/tests/test_api.py::test_check_food_not_found
PASSED app/tests/test_api.py::test_check_food_user_not_found
PASSED app/tests/test_api.py::test_check_food_typo
PASSED app/tests/test_api.py::test_ocr_normal
PASSED app/tests/test_api.py::test_ocr_allergy_trigger
PASSED app/tests/test_api.py::test_ocr_empty_text
PASSED app/tests/test_api.py::test_image_valid
PASSED app/tests/test_api.py::test_image_low_confidence
PASSED app/tests/test_api.py::test_image_unknown_food
PASSED app/tests/test_api.py::test_history_records_check
PASSED app/tests/test_api.py::test_history_detail
PASSED app/tests/test_api.py::test_history_delete
19 passed in X.XXs
```

## Step 7 · pgAdmin — inspect your database

Open http://localhost:5050

Login:
- Email: `admin@foodapp.com`
- Password: `admin123`

Add server connection:
1. Right-click **Servers** → **Register** → **Server**
2. Name: `FoodApp`
3. Connection tab:
   - Host: `postgres`
   - Port: `5432`
   - Database: `food_recommendation`
   - Username: `foodapp`
   - Password: `foodapp123`
4. Click Save

Browse:
- `Databases → food_recommendation → Schemas → public → Tables`
- `user_profiles` → right-click → View/Edit Data → All Rows
- `food_checks`   → same

You should see the rows created by the API calls in Step 5.

## Phase 2 Complete ✓

**What's working now:**

| Component | Status |
|-----------|--------|
| FastAPI server | ✓ running on :8000 |
| PostgreSQL | ✓ running on :5432 |
| Swagger UI | ✓ http://localhost:8000/docs |
| ML model | ✓ trained + loaded |
| Food lookup | ✓ 590 foods searchable |
| OCR pipeline | ✓ Tesseract running in Docker |
| User profiles | ✓ saved to PostgreSQL |
| Food check history | ✓ saved to PostgreSQL |
| All 3 input modes | ✓ manual / OCR / image |
| Tests | ✓ 19 passing |

**Next: Phase 3** — React Native / Expo frontend that connects to this API.